# Step 4 — Exact feature overlap and label consistency

This notebook **only audits** the officially designated UNSW-NB15 CSV files verified in Step 3. It uses the saved source file SHA-256 hashes, generates cryptographic SHA-256 signatures of the *42 parsed feature values* (excluding IDs/labels), and counts exact cross-split overlaps and contradictory labels.

**Research safeguard:** identical features are not automatically illicit leakage or evidence of shared original traffic events. We do not remove rows or alter the official test set. The lack of reliable timestamps still prevents genuine chronological guarantees.

Kaggle settings: **Internet enabled**; repository access via existing Kaggle secret `pushRare` (fine-grained GitHub Contents read/write). Never paste or print the token.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path("/kaggle/working/rare-attack-ids-step04")
URL = "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)

print("Git commit:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
MANIFEST = REPO / "data" / "manifests" / "unsw_nb15" / "audit_summary.json"
OUT = REPO / "data" / "manifests" / "unsw_nb15" / "exact_overlap"

subprocess.run([
    sys.executable, str(REPO / "scripts" / "exact_overlap_audit.py"),
    "--manifest", str(MANIFEST), "--output", str(OUT)],
    cwd=str(REPO), check=True)

print((OUT / "overlap_report.md").read_text(encoding="utf-8"))

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("pushRare")
except Exception:
    token = None

if not token:
    print("Audit complete but NOT SYNCED: grant this notebook access to Kaggle secret pushRare.")
else:
    safe_env = os.environ.copy()
    safe_env["GITHUB_TOKEN"] = token
    try:
        subprocess.run([
            sys.executable, str(REPO / "scripts" / "sync_exact_overlap.py"),
            "--directory", str(OUT)], cwd=str(REPO), env=safe_env, check=True)
    finally:
        safe_env.pop("GITHUB_TOKEN", None)
        del token

## What to share

Send the last-cell line beginning **VERIFIED EXACT-OVERLAP SYNC COMMIT**. We will read the actual aggregate results on GitHub and interpret the duplicate-label and chronology issues. No model training is performed.